<a href="https://colab.research.google.com/github/saskiaalifah/SaskiaAlifah_2411531002_BigData26/blob/main/Praktikum2/BD_A_T02_2411531002_SaskiaAlifah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Latihan Praktikum 2 — Pengumpulan dan Pra-pemrosesan Data
### Data Acquisition & Preprocessing — S1 Informatika





In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.4 MB/s eta 0:00:00


### Latihan 1: Ubah `SEED` menjadi 7 dan bandingkan jumlah baris

## K-0. Menyambungkan Google Drive dan Menyiapkan Folder Kerja



In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
    DIR_KERJA  = "/content/data"
except ModuleNotFoundError:
    # Lingkungan non-Colab (mis. Jupyter lokal): gunakan folder lokal yang setara
    print("Bukan lingkungan Colab -> memakai folder lokal sebagai pengganti Google Drive.")
    DIR_SIMPAN = "./drive_MyDrive_BigData_Praktikum2"
    DIR_KERJA  = "./data"

import os
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print("DIR_KERJA :", DIR_KERJA)
print("DIR_SIMPAN:", DIR_SIMPAN)
print(os.listdir(DIR_SIMPAN))

Mounted at /content/drive
DIR_KERJA : /content/data
DIR_SIMPAN: /content/drive/MyDrive/BigData/Praktikum2
['transaksi_bersih_42.csv', 'transaksi_mentah_42.csv', 'transaksi_bersih_7.csv', 'transaksi_mentah_7.csv']


**Penjelasan:** `DIR_KERJA` dipakai untuk berkas sementara selama proses berjalan, sedangkan
`DIR_SIMPAN` menunjuk ke folder **`BigData/Praktikum2`** di Google Drive — folder inilah yang menjadi
tujuan akhir `transaksi_mentah_7.csv` (K-2) dan `transaksi_bersih_7.csv` (K-6) di bawah, supaya kedua file
benar-benar tersimpan ke Drive dan bisa dibaca ulang oleh Praktikum 3. Blok `try/except` menjaga notebook
tetap bisa dijalankan di luar Colab (mis. Jupyter lokal) dengan folder lokal sebagai pengganti.

## K-1. Import Library dan Inisialisasi

Tujuan: menyiapkan seluruh pustaka yang dipakai sepanjang praktikum ini.

In [3]:
import numpy as np
import pandas as pd
from faker import Faker
import random

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)

pandas version: 2.2.3
numpy version : 2.1.3


**Penjelasan:** empat pustaka di atas menangani peran berbeda: `numpy` untuk operasi numerik dan
pembangkit bilangan acak, `pandas` untuk struktur data tabular (DataFrame), `Faker` untuk membangkitkan
data palsu (nama, kota, tanggal) yang realistis, dan `random` untuk pemilihan acak murni Python
(mis. `random.choice`). Sel di atas juga mencetak versi `pandas`/`numpy` yang benar-benar dipakai saat
notebook ini dijalankan — berguna untuk melacak jika ada perbedaan hasil dibanding modul, sesuai catatan
pada Bagian H modul.

## K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

Kode berikut mensimulasikan proses *acquisition* data transaksi, dengan sengaja menyisipkan variasi format harga, tanggal, kapitalisasi, *missing value*, dan baris *duplicate*.

In [4]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah_7.csv", index=False)


jumlah_mentah = len(df)
path_mentah = os.path.join(DIR_SIMPAN, "transaksi_mentah_7.csv")
df.to_csv(path_mentah, index=False)
print("Jumlah baris:", jumlah_mentah)
print("Tersimpan ke:", path_mentah)

Jumlah baris: 515
Tersimpan ke: /content/drive/MyDrive/BigData/Praktikum2/transaksi_mentah_7.csv


**Interpretasi (angka aktual hasil eksekusi):**

Sel di atas benar-benar mencetak **`Jumlah baris: 515`**. Angka ini berasal dari `N = 500` baris data
transaksi awal yang dibangkitkan pada loop, ditambah **15 baris duplikat** yang sengaja disisipkan
melalui `df.sample(n=15, ...)` lalu digabung dengan `pd.concat`, sehingga totalnya `500 + 15 = 515` baris.
File `transaksi_mentah.csv` disimpan langsung ke `DIR_SIMPAN` (folder `BigData/Praktikum2` di Google
Drive), bukan ke penyimpanan sementara runtime, sehingga tetap ada meski sesi Colab berakhir. Dataset
mentah ini juga sudah membawa *missing value* pada kolom `customer_name`, `shipping_city`, dan
`payment_method` karena baris-barisnya sudah di-set `NaN` sebelum proses duplikasi dijalankan — artinya
ada kemungkinan sebagian baris duplikat ikut membawa nilai kosong yang sama, yang akan terlihat
pengaruhnya pada Langkah K-3 dan K-4.

## K-3. Deteksi dan Penanganan Missing Value

In [5]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


**Interpretasi (angka aktual hasil eksekusi):** Dari **515 baris data mentah**, terdapat nilai kosong pada beberapa kolom, yaitu `customer_name` sebanyak **20**, `payment_method` sebanyak **16**, `shipping_city` sebanyak **30**, dan `rating` sebanyak **120**. Sementara itu, kolom `transaction_id`, `product_name`, `category`, `price`, `quantity`, dan `transaction_date` tidak memiliki nilai kosong karena masing-masing memiliki **0 missing value**.


In [6]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

**Interpretasi (angka aktual hasil eksekusi):** `dropna(subset=["customer_name","payment_method"])`
membuang **20 baris** — jumlah ini tepat sama dengan jumlah *missing value* pada `customer_name` (20),
yang berarti seluruh 16 baris yang kosong pada `payment_method` kebetulan sudah tercakup di dalam
20 baris yang dibuang tersebut (baris yang kosong di kedua kolom sekaligus). Jumlah baris turun dari
**515 menjadi 495**. Setelah itu, `fillna("Tidak Diketahui")` mengisi **30 baris** pada kolom
`shipping_city` tanpa membuang satu baris pun, sehingga informasi baris tersebut tetap bisa dipakai
untuk analisis kolom lain.

## K-4. Deteksi dan Penanganan Duplicate

In [7]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


**Interpretasi (angka aktual hasil eksekusi):** df.duplicated() menunjukkan terdapat 5 baris yang merupakan duplikat secara keseluruhan. Sementara transaction_id.duplicated() juga menunjukkan 5 ID transaksi berulang. Setelah drop_duplicates(), jumlah data berkurang dari 495 menjadi 490 baris.

## K-5. Koreksi Tipe Data dan Standardisasi Format

### a. Standardisasi Teks Kategorikal (`category`, `payment_method`, `shipping_city`)

In [8]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

**Interpretasi (angka aktual hasil eksekusi):** sebelum distandardisasi, kolom `category` memiliki
**12 variasi nilai unik** (mis. `"Buku"`, `"BUKU  "`, dsb — kombinasi huruf besar/kecil dan spasi ekstra
untuk 6 kategori dasar), dan `payment_method` memiliki **8 variasi nilai unik** (4 metode bayar dasar,
masing-masing punya versi huruf kecil karena `metode.lower()` diterapkan pada ~30% baris di K-2).
Setelah `.str.strip().str.title()` dan koreksi khusus `"Cod"` → `"COD"` dijalankan, kedua kolom
langsung turun tepat menjadi **6 nilai unik** untuk `category` (`Buku`, `Elektronik`, `Fashion`,
`Kesehatan`, `Olahraga`, `Rumah Tangga`) dan **4 nilai unik** untuk `payment_method`
(`COD`, `E-Wallet`, `Kartu Kredit`, `Transfer Bank`) — jumlah unik ini sekarang persis sama dengan
jumlah kategori/metode bayar asli yang didefinisikan di K-2.

### b. Koreksi Tipe Data pada Kolom `price` (dari teks bercampur simbol, menjadi numerik)

In [9]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

**Interpretasi (angka aktual hasil eksekusi):** 6 contoh nilai `price` yang tercetak sebelum
dibersihkan menunjukkan campuran format seperti `" 75000 "` (ada spasi), `"Rp500.000"` (ada prefix
`Rp` dan titik ribuan), dan `"1200000.0"` (ada desimal `.0`). Setelah melewati `bersihkan_harga()`,
seluruh nilai tersebut berhasil dikonversi menjadi angka murni bertipe numerik, misalnya `"Rp500.000"`
menjadi `500000.0`. Yang terpenting, **`0` baris** `price` gagal dikonversi (tidak ada nilai kosong
baru yang muncul akibat proses pembersihan) — artinya seluruh 8 varian format harga yang sengaja
disisipkan di K-2 berhasil ditangani oleh fungsi ini.

### c. Standardisasi Format Tanggal ke `YYYY-MM-DD`

**Kesalahan umum yang harus dihindari:** kode `pd.to_datetime(df["transaction_date"], format="mixed", dayfirst=True)`
**SALAH** untuk dipakai di sini. Kombinasi `format="mixed"` dengan `dayfirst=True` akan ikut "membalik"
tanggal yang sebenarnya sudah dalam format ISO (`YYYY-MM-DD`) yang tidak ambigu — misalnya `2026-07-11`
bisa salah terbaca menjadi `2026-11-07`. Solusi yang aman adalah mencoba format eksplisit satu per satu
untuk setiap nilai, seperti pada fungsi `parse_tanggal()` di bawah.

In [10]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

**Interpretasi (angka aktual hasil eksekusi):** contoh nilai sebelum standardisasi menampilkan tiga
format berbeda tercampur, misalnya `"11/07/2026"` (DD/MM/YYYY) dan `"2026-07-07"` (ISO). Setelah
`parse_tanggal()` diterapkan, `"11/07/2026"` berhasil dikonversi menjadi `"2026-07-11"` — **tanggal
11 Juli, bukan 7 November** — karena fungsi ini mencoba format `%d/%m/%Y` secara eksplisit, bukan
menebak arah hari/bulan seperti risiko pada kombinasi `format="mixed"` + `dayfirst=True` yang
diperingatkan modul. Hasil akhirnya, **`0` baris** tanggal gagal di-parse (`NaT`), yang berarti seluruh
3 varian format tanggal dari K-2 berhasil dikenali.

### d. Finalisasi Tipe Data

In [11]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

**Interpretasi (angka aktual hasil eksekusi):** setelah `astype(int)` dan `astype(float)` dijalankan,
kolom `quantity` dan `price` sekarang benar-benar bertipe numerik (`int64` dan `float64`), bukan lagi
teks. Kolom `category`, `payment_method`, dan `shipping_city` bertipe `string` hasil standardisasi
K-5a, sementara `transaction_id`, `product_name`, dan `transaction_date` tetap bertipe teks karena
memang dipakai sebagai label/identitas, bukan untuk operasi aritmatika.

## K-6. Ekspor Dataset Bersih

In [12]:
path_bersih = os.path.join(DIR_SIMPAN, "transaksi_bersih_7.csv")
df.to_csv(path_bersih, index=False)
jumlah_bersih = len(df)
print("Dataset bersih tersimpan:", jumlah_bersih, "baris")
print("Tersimpan ke:", path_bersih)

Dataset bersih tersimpan: 490 baris
Tersimpan ke: /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih_7.csv


**Interpretasi (angka aktual hasil eksekusi):** file `transaksi_bersih.csv` yang tersimpan berisi
**490 baris**, turun dari 515 baris data mentah di K-2 — selisih **25 baris (515 − 490 = 25)** hilang
sepanjang seluruh pipeline pra-pemrosesan (K-3 dan K-4). File ini tersimpan ke `DIR_SIMPAN`
(folder `BigData/Praktikum2` di Google Drive) berdampingan dengan `transaksi_mentah.csv`, sesuai yang
tercetak pada `os.listdir(DIR_SIMPAN)` di atas — bukti bahwa kedua *deliverable* CSV memang sudah
berada di Drive, bukan hanya di runtime sementara.

**Interpretasi (angka aktual hasil eksekusi):** hasil run menunjukkan bahwa `SEED=42` dan `SEED=7` sama-sama menghasilkan **515 baris mentah**, **495 baris setelah `dropna()`**, dan **490 baris setelah `drop_duplicates()`**. Jumlah 515 baris mentah berasal dari `N = 500` data awal ditambah `15` baris duplikat, sehingga jumlah tersebut tetap meskipun nilai `SEED` diubah. Sementara itu, perubahan `SEED` digunakan untuk mengatur proses acak dalam pembangkitan data, sehingga nilai atau isi baris yang dihasilkan dapat berbeda. Pada percobaan ini, kedua SEED menghasilkan jumlah baris yang sama setelah proses pembersihan, yaitu **490 baris**.


### Latihan 2: Tambahkan kolom `is_valid_price` (bernilai `True` jika `price > 0`)

In [13]:
df["is_valid_price"] = df["price"] > 0

print(df["is_valid_price"].value_counts())
print()
print("Harga minimum pada dataset bersih:", df["price"].min())
print("Jumlah baris dengan harga TIDAK valid (price <= 0):", (~df["is_valid_price"]).sum())

is_valid_price
True    490
Name: count, dtype: int64

Harga minimum pada dataset bersih: 15000.0
Jumlah baris dengan harga TIDAK valid (price <= 0): 0


**Interpretasi (angka aktual hasil eksekusi):** `value_counts()` menunjukkan seluruh **490 baris**
bernilai `True` pada `is_valid_price`, dan **0 baris** bernilai `False`. Harga minimum pada dataset
bersih adalah **15.000** (sesuai nilai `harga_dasar` terkecil yang didefinisikan di K-2). Artinya tidak
ditemukan satu pun harga tidak valid (≤ 0) pada dataset yang sudah melewati proses pembersihan K-5b —
fungsi `bersihkan_harga()` berhasil mengonversi seluruh varian format harga menjadi angka positif yang
wajar.

### Latihan 3: Hitung jumlah transaksi per `category` menggunakan `value_counts()`

In [14]:
jumlah_per_kategori = df["category"].value_counts()
print(jumlah_per_kategori)
print()
print("Total (harus sama dengan jumlah baris bersih):", jumlah_per_kategori.sum())

category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64

Total (harus sama dengan jumlah baris bersih): 490


**Interpretasi (angka aktual hasil eksekusi):** Berdasarkan hasil value_counts(), kategori dengan jumlah transaksi terbanyak adalah Rumah Tangga sebanyak 91 transaksi, diikuti Kesehatan sebanyak 86 transaksi, Buku sebanyak 81 transaksi, Fashion sebanyak 80 transaksi, Elektronik sebanyak 78 transaksi, dan Olahraga sebanyak 74 transaksi. Total keenam kategori tersebut adalah 490 transaksi, sama dengan jumlah baris pada dataset bersih. Hal ini menunjukkan bahwa seluruh baris memiliki nilai category dan tidak terdapat missing value pada kolom tersebut.